# Fashion Image Retrieval — Colab 실행 노트북

이미지 임베딩으로 **같은 상품 찾기** (DeepFashion In-Shop).
Baseline(zero-shot) → 실패 분석 → 개선 실험 순서로 위에서부터 실행하세요.

> ⚠️ **먼저 런타임 → 런타임 유형 변경 → 하드웨어 가속기 = T4 GPU** 로 설정!

## 0. GPU 확인 + 패키지 설치

In [ ]:
!nvidia-smi -L
!pip -q install faiss-cpu transformers datasets kagglehub
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. 코드(`src/`) 가져오기
본인 GitHub 레포 URL로 바꾸세요. (아직 레포가 없으면, 왼쪽 파일창에 `src/` 폴더를 업로드해도 됩니다.)

In [ ]:
import os, sys
REPO_URL = "https://github.com/YOUR_ID/fashion-image-retrieval.git"   # ← YOUR_ID만 본인 계정으로 교체
if not os.path.exists('fashion-image-retrieval'):
    !git clone $REPO_URL
%cd /content/fashion-image-retrieval
sys.path.append('/content/fashion-image-retrieval')

## 2. Google Drive 마운트 (임베딩 캐시 저장)
세션이 끊겨도 임베딩(.npy)은 Drive에 남아 재평가가 순식간입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/retrieval_cache'
os.makedirs(CACHE, exist_ok=True)
print('cache:', CACHE)

## 3. In-Shop 데이터 다운로드 (Kaggle, 자동)
- 최초 1회 Kaggle 토큰 필요: kaggle.com → 우측상단 프로필 → **Settings → API → Create New API Token** → 받은 `kaggle.json`을 Colab 파일창에 업로드하거나 아래가 뜨면 붙여넣기.
- 손으로 파일 옮길 필요 없이, 이 셀이 받아서 압축까지 풉니다 (~7GB, 몇 분).

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download("hserdaraltan/deepfashion-inshop-clothes-retrieval")
print('DATA_ROOT =', DATA_ROOT)

## 4. 데이터 로드 + 서브샘플
무료 Colab에 맞춰 상품 수를 줄입니다 (숫자는 자유롭게 조정).
- `query`: 500개 상품
- `gallery`: 최대 8,000장 (정답은 유지, distractor만 무작위)

In [ ]:
from src import data
splits = data.load_inshop(DATA_ROOT)
query, gallery, train = splits['query'], splits['gallery'], splits['train']
print('raw  ->  query:%d  gallery:%d  train:%d' % (len(query), len(gallery), len(train)))

query   = data.subsample_items(query, n_items=500, seed=0)
gallery = data.cap_gallery(query, gallery, max_gallery=8000, seed=0)
print('use  ->  query:%d  gallery:%d' % (len(query), len(gallery)))

## 5. Baseline 임베딩 — DINOv2 (zero-shot)
검색용 self-supervised 백본. CLS 토큰을 L2 정규화해서 사용.

In [ ]:
import numpy as np
from src import embed
E = embed.Embedder('dinov2')
g_vecs = E.encode(gallery.paths, batch_size=64, desc='gallery')
q_vecs = E.encode(query.paths,   batch_size=64, desc='query')
np.save(f'{CACHE}/g_dinov2.npy', g_vecs)
np.save(f'{CACHE}/q_dinov2.npy', q_vecs)
print('embeddings:', g_vecs.shape, q_vecs.shape)

## 6. FAISS 검색 + Recall@k (baseline 숫자)

In [ ]:
from src import metrics
nrel = metrics.n_relevant_per_query(query.item_ids, gallery.item_ids)
sims, idx = metrics.search(g_vecs, q_vecs, topk=50)
base = metrics.evaluate(idx, query.item_ids, gallery.item_ids, ks=(1,5,10), n_relevant=nrel)
results = {'dinov2_zeroshot': base}
print('DINOv2 zero-shot:', {k: round(v,3) for k,v in base.items()})

## 7. ★ 실패 갤러리 — 왜 틀렸나 (이 프로젝트의 심장)
top-5에 정답이 없는 query를 그리드로 저장하고 자동 태깅.
- `same_category_confusion`: 같은 카테고리인데 다른 상품 (색/형태 과의존 의심)
- `cross_category`: 아예 다른 카테고리 (배경/전역특징에 끌림 의심)

**저장된 이미지를 직접 눈으로 보고** 유형을 확정하세요. 그게 다음 개선의 근거가 됩니다.

In [ ]:
from src import failures
tags, n_fail = failures.build_gallery(idx, sims, query, gallery,
                                      out_dir='results/failures', k=5, n=60)
print(f'실패 query {n_fail}개 / 자동 태그 분포:', tags)

from IPython.display import Image as IPImage, display
import glob
for p in sorted(glob.glob('results/failures/*.png'))[:6]:
    display(IPImage(p))

---
# Day 2 — 개선 실험
각 실험을 baseline과 같은 프로토콜로 재평가하고 채택/기각을 수치로 남깁니다.

## 8. 백본 비교 — CLIP / ResNet50 (가설: self-sup 백본이 검색에 유리)

In [ ]:
for bk in ['clip', 'resnet50']:
    Eb = embed.Embedder(bk)
    gv = Eb.encode(gallery.paths, desc=f'{bk} gallery')
    qv = Eb.encode(query.paths,   desc=f'{bk} query')
    _, ix = metrics.search(gv, qv, topk=50)
    results[f'{bk}_zeroshot'] = metrics.evaluate(ix, query.item_ids, gallery.item_ids,
                                                 ks=(1,5,10), n_relevant=nrel)
    print(bk, {k: round(v,3) for k,v in results[f'{bk}_zeroshot'].items()})
    del Eb

## 9. 전처리 실험 — center-crop (가설: 배경 제거로 상품에 집중)

In [ ]:
gc = E.encode(gallery.paths, center_crop=True, desc='gallery crop')
qc = E.encode(query.paths,   center_crop=True, desc='query crop')
_, ixc = metrics.search(gc, qc, topk=50)
results['dinov2_centercrop'] = metrics.evaluate(ixc, query.item_ids, gallery.item_ids,
                                                ks=(1,5,10), n_relevant=nrel)
print('center-crop:', {k: round(v,3) for k,v in results['dinov2_centercrop'].items()})

## 10. Fine-tune — projection head (가설: 도메인 적응으로 검색 향상)
얼린 DINOv2 임베딩 위에 작은 head를 supervised-contrastive로 학습.
train 상품 1,000개만 사용 → 몇 분이면 수렴.

In [ ]:
from src import finetune
train_s = data.subsample_items(train, n_items=1000, seed=0)
tr_vecs = E.encode(train_s.paths, batch_size=64, desc='train')
np.save(f'{CACHE}/train_dinov2.npy', tr_vecs)

head = finetune.train_head(tr_vecs, train_s.item_ids, dim=tr_vecs.shape[1],
                           P=16, K=4, steps=600, lr=1e-3, temp=0.1)
g_ft = finetune.apply_head(head, g_vecs)
q_ft = finetune.apply_head(head, q_vecs)
_, ixf = metrics.search(g_ft, q_ft, topk=50)
results['dinov2_finetune'] = metrics.evaluate(ixf, query.item_ids, gallery.item_ids,
                                              ks=(1,5,10), n_relevant=nrel)
print('fine-tune:', {k: round(v,3) for k,v in results['dinov2_finetune'].items()})

## 11. 결과 표 저장 (→ README/포폴에 붙이기)

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[['recall@1','recall@5','recall@10','mAP@10']].round(4)
os.makedirs('results', exist_ok=True)
df.to_markdown('results/metrics.md')
df.to_csv('results/metrics.csv')
print(df)
print('\n저장: results/metrics.md, results/failures/*.png')
print('이제 로컬에서 git add results/ 후 커밋/푸시하세요 (커밋은 본인 계정으로!).')